In [10]:
import pandas as pd
import pyarrow as pa
import numpy as np
import openpyxl
import subprocess
import os

strains = ['BL21', 'Keio']
atacama_base = 'http://atacama.qb3.berkeley.edu/~mprice/feba/cgi/tnseq.cgi?orgId='
base = 'https://morgannprice.org/FEBA/'

# Downloading genome info

In [ ]:
"""
for s in strains:
    # check if ../barseq_browser/data/{s} exists, if not create it
    if not os.path.exists(f'../barseq_browser/data/{s}'):
        os.makedirs(f'../barseq_browser/data/{s}')
    if not os.path.exists(f'../../../PhageDataServer/data/{s}'):
        os.makedirs(f'../../../PhageDataServer/data/{s}')
    # download genome file with subprocess
    subprocess.run(['wget', atacama_base + s + '&file=genome.fna', '-O', f'../barseq_browser/data/{s}/genome.fna'])
    subprocess.run(['wget', atacama_base + s + '&file=aaseq', '-O', f'../barseq_browser/data/{s}/aaseq'])
    subprocess.run(['wget', atacama_base + s + '&file=genes.tab', '-O', f'../barseq_browser/data/{s}/genes.tab'])
""";

## Adding ecocyc data

In [ ]:
keio_e_1 = pd.read_csv('../../../Moriniere_2026_Metadata/ecocyc_common_Keio.txt', sep='\t')
keio_e_2 = pd.read_csv('../../../Moriniere_2026_Metadata/ecocyc_ids_Keio.txt', sep='\t')
keio_e_1 = keio_e_1[['Gene Name', 'Accession-1', 'Product']].rename(columns={'Accession-1': 'sysName', 'Gene Name': 'name'})
keio_e_2 = keio_e_2[['Gene Name', 'Accession-1']].rename(columns={'Accession-1': 'sysName', 'Gene Name': 'ecocyc_id'})
keio_e = keio_e_1[keio_e_1['sysName'].notnull()].merge(keio_e_2[keio_e_2['sysName'].notnull()], on='sysName', how='inner')
keio_e

,name,sysName,Product,ecocyc_id
0,yqjC,b3097,DUF1090 domain-containing protein YqjC,G7611
1,yqgA,b2966,DUF554 domain-containing protein YqgA,G7534
2,yqaA,b2689,DedA family protein YqaA,G7407
3,metN,b0199,L-methionine/D-methionine ABC transporter ATP ...,EG11621
4,ydgD,b1598,putative serine protease YdgD,G6856
...,...,...,...,...
4689,allC,b0516,allantoate amidohydrolase,G6285
4690,glaR,b2664,DNA-binding transcriptional repressor GlaR,EG12386
4691,pdeF,b2503,cyclic di-GMP phosphodiesterase PdeF,G7314
4692,dppC,b3542,dipeptide ABC transporter membrane subunit DppC,EG12626


In [ ]:
BL21_e_1 = pd.read_csv('../../../Moriniere_2026_Metadata/ecocyc_common_BL21.txt', sep='\t')
BL21_e_2 = pd.read_csv('../../../Moriniere_2026_Metadata/ecocyc_ids_BL21.txt', sep='\t')
BL21_e_1 = BL21_e_1[['All-Genes', 'Accession-2', 'Product']].rename(columns={'Accession-2': 'sysName', 'All-Genes': 'name'})
BL21_e_2 = BL21_e_2[['All-Genes', 'Accession-2']].rename(columns={'Accession-2': 'sysName', 'All-Genes': 'ecocyc_id'})
BL21_e = BL21_e_1[BL21_e_1['sysName'].notnull()].merge(BL21_e_2[BL21_e_2['sysName'].notnull()], on='sysName', how='inner')
BL21_e

,name,sysName,Product,ecocyc_id
0,aaeA,ECD_03101,aromatic carboxylic acid efflux pump membrane ...,ECD_RS16140
1,aaeB,ECD_03100,aromatic carboxylic acid efflux pump subunit AaeB,ECD_RS16135
2,aaeR,ECD_03103,DNA-binding transcriptional activator AaeR,ECD_RS16150
3,aaeX,ECD_03102,DUF1656 domain-containing protein AaeX,ECD_RS16145
4,aas,ECD_02684,fused 2-acylglycerophospho-ethanolamine acyltr...,ECD_RS14015
...,...,...,...,...
4130,zraR,ECD_03881,DNA-binding transcriptional activator ZraR,ECD_RS20385
4131,zraS,ECD_03880,sensor histidine kinase ZraS,ECD_RS20380
4132,zupT,ECD_02912,divalent metal ion transporter ZupT,ECD_RS15150
4133,zur,ECD_03918,DNA-binding transcriptional repressor Zur,ECD_RS20620


In [ ]:
ecocyc_data = {'BL21': BL21_e, 'Keio': keio_e}
for s in strains:
    td = pd.read_csv(f'../barseq_browser/data/{s}/genes.tab', sep='\t').rename(columns={'name': 'original_name'})
    td = td.merge(ecocyc_data[s], on='sysName', how='left')
    td.to_csv(f'../barseq_browser/data/{s}/genes_w_ecocyc.tab', sep='\t', index=False)

## RB-TnSeq gene fitness etc.
* Also doing strain counts here and adding a filter to the RB-TnSeq data:
  * Filter columns are like set#_num_filtered_insertions = # insertions in the central 10–90% of the gene with >=5 reads in the sample
  * Then I will filter in the browser if the gene has >=3 filtered insertions

In [ ]:
# Get sets used from metadata
meta = pd.read_excel('../../../Moriniere_2026_Metadata/RBTnseq_sets_final.xlsx')
sets_used = dict()
for s in strains:
    s_meta = meta[meta['nickname']==s]
    all_sets_nested = [[x] for x in s_meta['set']]+[x.split(';') for x in s_meta['time0']]+[x.split(';') for x in s_meta['no_phage_control'] if not pd.isnull(x)]
    # flatten
    sets_used[s] = set([item for sublist in all_sets_nested for item in sublist])
meta.to_csv('../barseq_browser/data/RBTnseq_sets.csv', index=False)

In [ ]:

strains = ['BL21', 'Keio']
icols = ['locusId', 'sysName', 'desc']
strain_count_icols = ['barcode', 'scaffold', 'strand', 'pos', 'locusId', 'f']
for s in strains:
    print('reading RBTnSeq data for', s)
    t = pd.read_csv(base+s+'/fit_t.tab', sep='\t')
    t = t.rename(columns={x: x.split()[0]+"_T" for x in t.columns if x.split()[0] in sets_used[s]})
    t = t[icols+sorted([i+'_T' for i in sets_used[s]])]
    f = pd.read_csv(base+s+'/fit_logratios.tab', sep='\t')
    f = f.rename(columns={x: x.split()[0] for x in f.columns if x.split()[0] in sets_used[s]})
    f = f[icols+sorted(sets_used[s])]
    # same for half1 and half2
    f1 = pd.read_csv(base+s+'/fit_logratios_half1.tab', sep='\t')
    f1 = f1.rename(columns={x: x.split()[0]+"_half1" for x in f1.columns if x.split()[0] in sets_used[s]})
    f1 = f1[icols+sorted([i+'_half1' for i in sets_used[s]])]
    f2 = pd.read_csv(base+s+'/fit_logratios_half2.tab', sep='\t')
    f2 = f2.rename(columns={x: x.split()[0]+"_half2" for x in f2.columns if x.split()[0] in sets_used[s]})
    f2 = f2[icols+sorted([i+'_half2' for i in sets_used[s]])]
    # merge
    merged = pd.merge(t, f, on=icols)
    merged = pd.merge(merged, f1, on=icols)
    merged = pd.merge(merged, f2, on=icols)
    print(len(merged.columns), len(set(merged.columns)), len(set([i.split()[0] for i in merged.columns])))
    # change the dtype of the columns to reduce memory usage
    merged[[i for i in merged if i not in icols]] = merged[[i for i in merged if i not in icols]].astype('float16')
    merged = merged.merge(pd.read_csv(f'../barseq_browser/data/{s}/genes_w_ecocyc.tab', sep='\t')[['locusId', 'scaffoldId', 'name', 'begin', 'end', 'strand', 'ecocyc_id']], on='locusId', how='left')
    
    print('reading strain counts for', s)
    # STRAIN COUNTS
    p = pd.read_csv(base+s+'/all.poolcount', sep='\t')
    p = p.rename(columns=lambda x: x.split('_')[-1].replace('.', ''))
    # export to parquet with the dtype
    p[strain_count_icols+sorted(sets_used[s])].to_parquet(f'../../../PhageDataServer/data/{s}/strain_counts.parquet')
    print('computing strain count filter for', s)
    strain_count_filter = p[(p.f>0.1) & (p.f<0.9)][['locusId']+sorted(sets_used[s])].groupby('locusId').agg(lambda x: (x>=5).sum()).reset_index()
    print('merging and exporting final RBTnSeq data for', s)
    merged.merge(strain_count_filter, on='locusId', how='left', suffixes=('', '_num_filtered_insertions')).to_feather(f'../barseq_browser/data/{s}/RBTnSeq.feather', compression='uncompressed')
    
    # STRAIN FITS
    f = pd.read_csv(base+s+'/strain_fit.tab', sep='\t')
    f = f.rename(columns=lambda x: x.split('_')[-1].replace('.', ''))
    f = f[strain_count_icols+sorted(sets_used[s])]
    f[sorted(sets_used[s])] = f[sorted(sets_used[s])].astype('float16')
    # export to parquet with the dtype
    f.to_parquet(f'../../../PhageDataServer/data/{s}/strain_fits.parquet')

reading RBTnSeq data for BL21
1003 1003 1003
reading strain counts for BL21


/var/folders/kk/9bz46_553xq5lsbj897t_rvw0000gn/T/ipykernel_6193/2679464051.py:30: DtypeWarning: Columns (3,5) have mixed types. Specify dtype option on import or set low_memory=False.
  p = pd.read_csv(base+s+'/all.poolcount', sep='\t')


computing strain count filter for BL21
merging and exporting final RBTnSeq data for BL21


/var/folders/kk/9bz46_553xq5lsbj897t_rvw0000gn/T/ipykernel_6193/2679464051.py:40: DtypeWarning: Columns (3,5) have mixed types. Specify dtype option on import or set low_memory=False.
  f = pd.read_csv(base+s+'/strain_fit.tab', sep='\t')


reading RBTnSeq data for Keio
2435 2435 2435
reading strain counts for Keio
computing strain count filter for Keio
merging and exporting final RBTnSeq data for Keio


## Metadata

In [ ]:
from collections import Counter
td = pd.read_excel('../../../Moriniere_2026_Metadata/RBTnseq_sets_final.xlsx')
phage_reps = []
phage_rep_d = {'Keio': Counter(), 'BL21': Counter()}
for idx, row in td.iterrows():
    nickname = row['nickname']
    phage_rep_d[nickname][row['phage']] += 1
    phage_reps.append(row['phage']+' R'+str(phage_rep_d[nickname][row['phage']]))

td['phage_rep'] = phage_reps
td.to_csv('../barseq_browser/data/RBTnseq_sets.csv', index=False)